# Van Loan Extended: Non-Constant Intensities & Discounting

This notebook builds on `van_loan_theorem.ipynb`, where we saw Van Loan with **constant** intensities and `expm`.

Here we tackle two realistic complications:

1. **Non-constant intensities** — rates like mortality depend on age, so Λ(t) varies with time. We can no longer use `expm`; instead we solve an ODE (Kolmogorov's equation) numerically using **Runge-Kutta 4 (RK4)**.
2. **Discount factors** — in actuarial work, money received at time u is worth less than money received today. We need to incorporate `e^{-r·u}` into our product integral.

The key insight: **Van Loan still works in both cases** — it reduces everything to a single ODE solve of a block matrix system, instead of nested loops.

---

## Recap: The Core Integral

We want to compute:

$$
\int_0^T P(0,u)\, B(u)\, P(u,T)\, \mathrm{d}u
$$

where P(s, t) satisfies **Kolmogorov's forward equation**:

$$
\frac{d}{dt} P(s,t) = P(s,t) \cdot \Lambda(t), \quad P(s,s) = I
$$

With constant Λ this gives `P(s,t) = expm(Λ(t-s))`. With time-varying Λ(t), **we must solve the ODE numerically**.

## Part 1: Non-Constant Intensities

### The Model: Age-Dependent Mortality (Makeham's Law)

A classic actuarial mortality model is **Makeham's law**:

$$
\mu(t) = \alpha + \beta \cdot e^{\gamma t}
$$

where t is age. Mortality increases exponentially with age — much more realistic than a constant rate.

We use the same 3-state disability model as before:
- **0**: Active
- **1**: Disabled
- **2**: Dead

But now the death intensities μ_a(t) and μ_d(t) follow Makeham's law and **vary with time**:

$$
\Lambda(t) = \begin{pmatrix} -(\sigma + \mu_a(t)) & \sigma & \mu_a(t) \\ \rho & -(\rho + \mu_d(t)) & \mu_d(t) \\ 0 & 0 & 0 \end{pmatrix}
$$

### Why We Need RK4

Kolmogorov's equation `dP/dt = P · Λ(t)` is now an ODE with a **time-varying coefficient**. There is no closed-form solution. We use **RK4** (Runge-Kutta order 4) to solve it numerically.

RK4 approximates the ODE `dy/dt = f(t, y)` using four slope estimates per step:

```
k1 = f(t,       y)
k2 = f(t + h/2, y + h/2 * k1)
k3 = f(t + h/2, y + h/2 * k2)
k4 = f(t + h,   y + h   * k3)
y_next = y + (h/6)(k1 + 2k2 + 2k3 + k4)
```

This gives **4th-order accuracy** — errors shrink as h⁴, much better than simple Euler steps.

In [ ]:
import numpy as np
from scipy.linalg import expm

# ── Makeham mortality parameters ──
alpha, beta, gamma_m = 0.0005, 0.00007, 0.09  # typical life-table scale

# Disability / recovery intensities (still constant for simplicity)
sigma = 0.05  # active -> disabled
rho   = 0.02  # disabled -> active

def makeham_mu(t):
    """Makeham mortality intensity at age t."""
    return alpha + beta * np.exp(gamma_m * t)

def Lambda(t):
    """Time-varying 3x3 intensity matrix at age t."""
    mu_a = makeham_mu(t)       # active -> dead
    mu_d = 1.5 * makeham_mu(t) # disabled have 50% higher mortality
    return np.array([
        [-(sigma + mu_a),  sigma,          mu_a],
        [ rho,            -(rho + mu_d),   mu_d],
        [ 0,               0,               0  ]
    ])

# Inspect at a few ages
T = 10   # time horizon (10 years)
t0 = 50  # starting age

print("Λ at age 50 (t=0):")
print(np.round(Lambda(t0), 5))
print("\nΛ at age 60 (t=10):")
print(np.round(Lambda(t0 + T), 5))
print("\nNote: mortality roughly doubles over 10 years!")

### RK4 Solver for the Transition Matrix

We solve `dP/dt = P · Λ(t)` from `t = t_start` to `t = t_end`, starting at `P(t_start, t_start) = I`.

The key design: the function **infers the matrix dimension** from the output of `Lambda_func`, so it works for both the 3×3 state system and the 6×6 Van Loan block system.

In [ ]:
def rk4_transition(Lambda_func, t_start, t_end, n_steps, t_offset=0.0):
    """
    Solve dΦ/dt = Φ @ Lambda_func(t_offset + t) from t_start to t_end using RK4.
    Initial condition: Φ(t_start, t_start) = I.
    
    Works for any square system: n×n (single Λ) or 2n×2n (Van Loan block).
    t_offset: global time shift (e.g. starting age 50).
    """
    # Infer dimension from the function itself — not from any outer variable
    dim = Lambda_func(t_offset + t_start).shape[0]
    P = np.eye(dim)  # initial condition: Φ(s,s) = I
    h = (t_end - t_start) / n_steps

    def f(t, P):
        return P @ Lambda_func(t_offset + t)

    t = t_start
    for _ in range(n_steps):
        k1 = f(t,       P)
        k2 = f(t + h/2, P + h/2 * k1)
        k3 = f(t + h/2, P + h/2 * k2)
        k4 = f(t + h,   P + h   * k3)
        P  = P + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)
        t += h

    return P


# Sanity check: compare RK4 to expm on a constant-Λ slice
P_rk4   = rk4_transition(Lambda, 0, T, n_steps=1000, t_offset=t0)
P_const = expm(Lambda(t0) * T)  # constant approximation using Λ at age 50 only

print("P(0,T) via RK4 (time-varying Λ):")
print(np.round(P_rk4, 4))
print("\nP(0,T) via expm(Λ(50)·T) — ignores age increase:")
print(np.round(P_const, 4))
print("\nDifference (constant Λ underestimates mortality):")
print(np.round(P_rk4 - P_const, 4))

### Method 1 (Hard Way): Brute-Force Double Loop

To compute the sandwich integral `∫₀ᵀ P(0,u) · B · P(u,T) du`:

- **Outer loop** over u (to numerically integrate over u)
- For each u: **inner RK4 solve** for P(0,u) **and** another for P(u,T)

This is the *double loop* mentioned earlier.

In [ ]:
n_states = 3
n_outer  = 300   # outer quadrature steps over u
n_inner  = 200   # RK4 steps per P(s,t) solve
dt = T / n_outer

# Benefit matrix: 1 unit/year while disabled (state 1)
B_matrix = np.diag([0, 1, 0])

sandwich_bf = np.zeros((n_states, n_states))

for k in range(n_outer):
    u = k * dt
    # Each of these is itself an inner loop of n_inner RK4 steps
    P_0u = rk4_transition(Lambda, 0, u, n_steps=n_inner, t_offset=t0) if u > 0 else np.eye(n_states)
    P_uT = rk4_transition(Lambda, u, T, n_steps=n_inner, t_offset=t0)
    sandwich_bf += P_0u @ B_matrix @ P_uT * dt

print("Brute-force double-loop result (∫ P(0,u) B P(u,T) du):")
print(np.round(sandwich_bf, 4))
print(f"\nTotal matrix ODE solves: {2 * n_outer} (two RK4 runs per u)")
print(f"Total RK4 steps:         {2 * n_outer * n_inner:,}")

### Method 2 (Van Loan): Single ODE Solve of Block System

Van Loan says: build the block matrix

$$
M(t) = \begin{pmatrix} \Lambda(t) & B \\ 0 & \Lambda(t) \end{pmatrix}
$$

and solve the **single ODE** on the 6×6 system:

$$
\frac{d}{dt} \Phi(t) = \Phi(t) \cdot M(t), \quad \Phi(0) = I_6
$$

The upper-right 3×3 block of Φ(T) gives exactly `∫₀ᵀ P(0,u) B P(u,T) du`.

This is **one** RK4 solve replacing hundreds of nested RK4 runs.

In [ ]:
def Lambda_block(t):
    """Build the 6×6 Van Loan block matrix: [[Λ(t), B], [0, Λ(t)]]."""
    L = Lambda(t)
    M = np.zeros((2 * n_states, 2 * n_states))
    M[:n_states, :n_states] = L         # upper-left:  A = Λ(t)
    M[:n_states, n_states:] = B_matrix  # upper-right: B (benefit matrix)
    M[n_states:, n_states:] = L         # lower-right: C = Λ(t)
    return M

# One RK4 solve on the 6×6 block system — rk4_transition auto-detects size = 6
Phi = rk4_transition(Lambda_block, 0, T, n_steps=2000, t_offset=t0)

# The sandwich integral is the upper-right 3×3 block
sandwich_vl = Phi[:n_states, n_states:]

print("Van Loan result (single 6×6 ODE solve):")
print(np.round(sandwich_vl, 4))

print("\nDifference vs brute force:")
print(np.round(sandwich_vl - sandwich_bf, 4))
print(f"Max absolute error: {np.max(np.abs(sandwich_vl - sandwich_bf)):.2e}")
print("\n(Small error is from brute-force quadrature, not Van Loan — Van Loan is the reference.)")

## Part 2: Discount Factors

In actuarial life insurance, payments at future time u are worth less today due to the **time value of money**. With a constant force of interest r, we discount by `e^{-r·u}`, so we want:

$$
\int_0^T e^{-r u}\, P(0,u)\, B\, P(u,T)\, \mathrm{d}u
$$

### How to Incorporate Discounting into Van Loan

Recall (Corollary 1.15): discounting by `e^{-ru}` can be absorbed into the product integral by shifting the intensity:

$$
e^{-r(t-s)} \cdot P(s,t) = \prod_s^t \left(I + (\Lambda(x) - r I)\, \mathrm{d}x\right)
$$

**Intuition:** the extra `−r·I` adds a continuous "leakage" rate r to every state, which is exactly what discounting does — it shrinks the value at rate r over time.

So to add discounting, we simply replace **Λ(t)** with **Λ(t) − r·I** everywhere in the Van Loan block matrix. No extra code needed!

In [ ]:
r = 0.04  # 4% annual force of interest

def Lambda_block_discounted(t):
    """Van Loan block matrix with discounting:
       M(t) = [[Λ(t) − r·I,  B], [0,  Λ(t) − r·I]]
    """
    L = Lambda(t) - r * np.eye(n_states)  # shifted intensity = discounting
    M = np.zeros((2 * n_states, 2 * n_states))
    M[:n_states, :n_states] = L
    M[:n_states, n_states:] = B_matrix
    M[n_states:, n_states:] = L
    return M

# Van Loan: single ODE solve with discounting
Phi_disc = rk4_transition(Lambda_block_discounted, 0, T, n_steps=2000, t_offset=t0)
discounted_vl = Phi_disc[:n_states, n_states:]

print("Discounted sandwich integral (Van Loan, r=4%):")
print(np.round(discounted_vl, 4))

# Verify with explicit brute force (loop + discount weight * dt)
discounted_bf = np.zeros((n_states, n_states))
for k in range(n_outer):
    u = k * dt
    P_0u = rk4_transition(Lambda, 0, u, n_steps=n_inner, t_offset=t0) if u > 0 else np.eye(n_states)
    P_uT = rk4_transition(Lambda, u, T, n_steps=n_inner, t_offset=t0)
    discounted_bf += np.exp(-r * u) * P_0u @ B_matrix @ P_uT * dt

print("\nBrute-force discounted result:")
print(np.round(discounted_bf, 4))
print(f"\nMax error vs brute force: {np.max(np.abs(discounted_vl - discounted_bf)):.2e}")

## Timing Comparison

In [ ]:
import time

# Time the brute-force double loop
start = time.perf_counter()
for _ in range(3):
    bf_tmp = np.zeros((n_states, n_states))
    for k in range(n_outer):
        u = k * dt
        P_0u = rk4_transition(Lambda, 0, u, n_steps=n_inner, t_offset=t0) if u > 0 else np.eye(n_states)
        P_uT = rk4_transition(Lambda, u, T, n_steps=n_inner, t_offset=t0)
        bf_tmp += np.exp(-r * u) * P_0u @ B_matrix @ P_uT * dt
t_bf = (time.perf_counter() - start) / 3

# Time Van Loan (single block ODE solve)
start = time.perf_counter()
for _ in range(50):
    Phi_tmp = rk4_transition(Lambda_block_discounted, 0, T, n_steps=2000, t_offset=t0)
    vl_tmp  = Phi_tmp[:n_states, n_states:]
t_vl = (time.perf_counter() - start) / 50

print(f"Brute force: {t_bf:.3f} seconds")
print(f"Van Loan:    {t_vl:.4f} seconds")
print(f"\nSpeedup: {t_bf / t_vl:.0f}x faster!")

## Summary

| Aspect | Constant Λ | Time-varying Λ(t) |
|--------|-----------|-------------------|
| **P(s,t) formula** | `expm(Λ·(t-s))` | RK4 ODE solve |
| **Brute-force loops** | O(N) `expm` calls | O(N) RK4 solves in outer loop + O(M) steps each = O(N·M) total |
| **Van Loan** | 1 `expm` call | 1 RK4 solve on the block system |
| **Adding discounting** | Replace Λ → Λ − r·I | Exact same trick, no extra cost |

**The Van Loan philosophy doesn't change** when Λ varies with time. The only difference is *how* we compute the product integral (expm vs. RK4). The block structure and the "read off the upper-right corner" trick remain identical.

---

## Check Your Understanding

1. In `rk4_transition`, what is the **initial condition** and why does it make mathematical sense?

2. We used `Λ(t) − r·I` to incorporate discounting. Think about what ODE `y(t) = e^{-rt}` satisfies — how does subtracting `r·I` from Λ achieve the discounting effect?

3. In the original notebook (constant Λ, sojourn time), we had **C = 0** in the Van Loan block. Here we use **C = Λ(t)**. What is the financial/probabilistic meaning of this change? (*Hint: what does P(u, T) represent in a life insurance reserve?*)